# Example 03 - 1D PLUMED FES + Hummer D(s)

This notebook mirrors `examples/03_1d_hummer_D_ctmc.py` using the
bundled synthetic FES and diffusion-profile CSV. It feeds a
position-dependent diffusion profile into the 1D CTMC workflow.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "stochkin").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "examples" / "data"
OUT_DIR = ROOT / "notebooks" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Notebook output directory: {OUT_DIR}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stochkin as sk

from stochkin.plotting import _apply_pub_axes, _apply_pub_cbar
from stochkin.style import publication_style


In [ ]:
fes_path = DATA_DIR / "synthetic_1d_fes.dat"
d_csv = DATA_DIR / "synthetic_diffusion_profile.csv"
crop = (0.5, 9.5)
T = 300.0
resample_n = 500
core_fraction = 0.05

result = sk.run_1d_ctmc_with_hummer_D(
    fes_path=fes_path,
    d_csv=d_csv,
    T=T,
    d_xcol="x_interface",
    d_col="D_med",
    d_grid="interface",
    crop=crop,
    resample_n=resample_n,
    core_fraction=core_fraction,
)


In [ ]:
basin_labels = [f"B{bid}" for bid in result["basin_ids"]]
K_ns = pd.DataFrame(
    result["K_ps"] * 1000.0,
    index=basin_labels,
    columns=basin_labels,
)
exit_times_ns = pd.Series(
    result["exit_ps"] / 1000.0,
    index=basin_labels,
    name="exit_time_ns",
)

K_ns


In [ ]:
exit_times_ns


In [ ]:
s = result["s"]
F = result["F"]
D_used = np.asarray(result["D_used"])

out_fes = OUT_DIR / "03_fes_and_D.png"
out_rates = OUT_DIR / "03_ctmc_rates.png"

with publication_style():
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.3, 4.2), sharex=True)
    ax1.plot(s, F, "k-", lw=1.5)
    _apply_pub_axes(ax1, ylabel=r"F  [kJ mol$^{-1}$]", title="Free-energy profile")

    ax2.plot(s, D_used, "b-", lw=1.5)
    _apply_pub_axes(
        ax2,
        xlabel="CV",
        ylabel=r"$D(s)$  [CV$^2$ ps$^{-1}$]",
        title="Hummer D(s) (interpolated)",
    )

    fig.tight_layout()
    fig.savefig(out_fes, dpi=300)

with publication_style():
    K_abs = np.abs(K_ns.to_numpy())
    with np.errstate(divide="ignore", invalid="ignore"):
        Klog = np.where(K_abs > 0, np.log10(K_abs), np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(6.6, 2.8))
    im = axes[0].imshow(Klog, cmap="magma_r", aspect="auto")
    axes[0].set_xticks(range(len(basin_labels)))
    axes[0].set_xticklabels(basin_labels)
    axes[0].set_yticks(range(len(basin_labels)))
    axes[0].set_yticklabels(basin_labels)
    cbar = fig.colorbar(im, ax=axes[0])
    _apply_pub_cbar(cbar, label=r"$\log_{10}|K_{ij}|$  [ns$^{-1}$]")
    _apply_pub_axes(axes[0], title="Rate matrix")

    axes[1].bar(range(len(basin_labels)), exit_times_ns.to_numpy(), color="steelblue")
    axes[1].set_yscale("log")
    axes[1].set_xticks(range(len(basin_labels)))
    axes[1].set_xticklabels(basin_labels)
    _apply_pub_axes(axes[1], ylabel="Exit time  [ns]", title="Mean exit times")

    fig.tight_layout()
    fig.savefig(out_rates, dpi=300)

print(f"Saved {out_fes.relative_to(ROOT)}")
print(f"Saved {out_rates.relative_to(ROOT)}")
plt.show()
